##### **문항1**

In [ ]:
import pandas as pd

rating = pd.read_csv('ratings_small.csv')
movieInfo = pd.read_csv('tmdb_5000_credits.csv')

movieInfo = movieInfo.rename(columns={'movie_id': 'movieId'})

df = pd.merge(rating, movieInfo, on='movieId', how='inner')

df = df[['movieId', 'userId', 'rating', 'title']]

df.head()	# 확인용

##### **문항2**

In [ ]:
import pandas as pd

rating = pd.read_csv('ratings_small.csv')
movieInfo = pd.read_csv('tmdb_5000_credits.csv')

movieInfo = movieInfo.rename(columns={'movie_id': 'movieId'})
df = pd.merge(rating, movieInfo, on='movieId', how='inner')

df = df.drop_duplicates(subset=['userId'])

df['user'] = df['userId'].astype('category').cat.codes

df['movie'] = df['movieId'].astype('category').cat.codes

df = df[['movieId', 'userId', 'rating', 'title', 'movie', 'user']]

df.head()

##### **문항3**

In [ ]:
import tensorflow as tf
import pandas as pd


rating = pd.read_csv('ratings_small.csv')
movieInfo = pd.read_csv('tmdb_5000_credits.csv')

movieInfo = movieInfo.rename(columns={'movie_id': 'movieId'})

df = pd.merge(rating, movieInfo, on='movieId', how='inner')

df['user'] = df['userId'].astype('int32')
df['movie'] = df['movieId'].astype('int32')
df['rating'] = df['rating'].astype('float32')

dataset = tf.data.Dataset.from_tensor_slices((
    {
        "user": df["user"].values,
        "movie": df["movie"].values
    },
    df["rating"].values
))

dataset = dataset.cache()
dataset = dataset.shuffle(100)
dataset = dataset.batch(32)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

print(dataset)

##### **문항4**

In [ ]:
from tensorflow.keras import layers, models

user_input = layers.Input(shape=(1,), name='user_in')
movie_input = layers.Input(shape=(1,), name='movie_in')


user_emb = layers.Embedding(input_dim=df['user'].nunique(), output_dim=16, name='user_emb')(user_input)

movie_emb = layers.Embedding(input_dim=df['movie'].nunique(), output_dim=16, name='movie_emb')(movie_input)

user_vec = layers.Flatten()(user_emb)
movie_vec = layers.Flatten()(movie_emb)

x = layers.Concatenate()([user_vec, movie_vec])

x = layers.Dense(32, activation='relu')(x)
x = layers.Dense(16, activation='relu')(x)

outputs = layers.Dense(1, activation='linear')(x)

model = models.Model(inputs=[user_input, movie_input], outputs=outputs)

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.fit(dataset, epochs=10)

In [ ]:
import numpy as np

user_id = 1

user_idx = df[df['userId'] == user_id]['user'].iloc[0]

movie_ids = df['movie'].unique()

user_array = np.array([user_idx] * len(movie_ids))
movie_array = np.array(movie_ids)

predictions = model.predict([user_array, movie_array])

recommend_df = pd.DataFrame({
    'movie': movie_ids,
    'pred_rating': predictions.flatten()
})

recommend_df = recommend_df.merge(
    df[['movie', 'title']].drop_duplicates(),
    on='movie'
)

top10 = recommend_df.sort_values(
    by='pred_rating',
    ascending=False
).head(10)

print(top10[['title', 'pred_rating']])